# 🚀 Leocherd Iris Energy — GPU 3D Motion & Rigging Reconstruction

Welcome to the official **Leocherd Iris Energy** cloud GPU acceleration pipeline.
This notebook runs on **free Kaggle GPUs (Tesla T4 / P100)** to perform high-fidelity, high-fps 3D human pose tracking, biomechanical regularization, and temporal motion smoothing on any video using the modern **MediaPipe 1.x Tasks API**.

### Workflow Overview:
1. **Upload Video**: Upload any `.mp4` or `.mov` clip to Kaggle (or attach as Dataset).
2. **GPU Inference**: Google MediaPipe Heavy Task model runs with GPU acceleration.
3. **Temporal Filtering**: Savitzky-Golay smoothing eliminates monocular jitter and Z-axis popping.
4. **Kinematic Calibration**: World coordinates are transformed into canonical Leocherd Iris skeleton hierarchy.
5. **Export & Download**: Produces `iris_kaggle_pose_data.json` ready for local 1-click 3D character rigging!


In [ ]:
# 1. Environment & Package Installation
!nvidia-smi

!pip install -q mediapipe opencv-python-headless scipy plotly
!wget -q -O pose_landmarker.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/latest/pose_landmarker_heavy.task
print('✅ Environment ready. Model downloaded.')


In [ ]:
# 2. Core Python Imports
import os
import glob
import json
import cv2
import numpy as np
import torch
from scipy.signal import savgol_filter
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ PyTorch running on: {device}')
if torch.cuda.is_available():
    print(f'   Device Name: {torch.cuda.get_device_name(0)}')


In [ ]:
# 3. Detect Uploaded Video (Both INPUT_VIDEO and VIDEO_PATH aliases defined)
videos = glob.glob('/kaggle/input/**/*.mp4', recursive=True) + glob.glob('/kaggle/working/*.mp4')
if not videos:
    raise RuntimeError('⚠️ No MP4 video found! Please upload a video or attach your dataset.')
INPUT_VIDEO = videos[0]
VIDEO_PATH = INPUT_VIDEO  # Alias for compatibility
print(f'📹 Found video: {INPUT_VIDEO}')


In [ ]:
# 4. Initialize MediaPipe Heavy Pose Landmarker with GPU Acceleration
# In MediaPipe Tasks, delegate=mp_tasks.BaseOptions.Delegate.GPU tells MediaPipe to use GPU.
# We try GPU first, with graceful fallback to high-speed LiteRT XNNPACK engine if headless Linux EGL is absent.
detector = None
try:
    print('Attempting GPU delegate initialization...')
    base_options = mp_tasks.BaseOptions(
        model_asset_path='pose_landmarker.task',
        delegate=mp_tasks.BaseOptions.Delegate.GPU   # <--- Tells MediaPipe to use GPU
    )
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.IMAGE,
        min_pose_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    detector = vision.PoseLandmarker.create_from_options(options)
    print('🚀 SUCCESS: MediaPipe 1.x Pose Landmarker initialized with GPU acceleration!')
except Exception as e:
    print(f'⚠️ GPU delegate unavailable ({e}). Initializing high-speed LiteRT XNNPACK multi-threaded engine...')
    base_options = mp_tasks.BaseOptions(model_asset_path='pose_landmarker.task')
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.IMAGE,
        min_pose_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    detector = vision.PoseLandmarker.create_from_options(options)
    print('✅ MediaPipe 1.x Pose Landmarker initialized successfully (LiteRT multi-threaded engine).')

LANDMARK_MAP = {
    0: 'Nose',
    7: 'LeftEar', 8: 'RightEar',
    11: 'LeftShoulder', 12: 'RightShoulder',
    13: 'LeftArm', 14: 'RightArm',
    15: 'LeftForeArm', 16: 'RightForeArm',
    17: 'LeftHand', 18: 'RightHand',
    23: 'LeftUpLeg', 24: 'RightUpLeg',
    25: 'LeftLeg', 26: 'RightLeg',
    27: 'LeftFoot', 28: 'RightFoot',
}


In [ ]:
# 5. Process Video & Extract Metric World Landmarks
video_path = globals().get('INPUT_VIDEO') or globals().get('VIDEO_PATH')
if not video_path:
    vids = glob.glob('/kaggle/input/**/*.mp4', recursive=True) + glob.glob('/kaggle/working/*.mp4')
    if not vids:
        raise RuntimeError('⚠️ No MP4 video found! Please attach your dataset or upload a video.')
    video_path = vids[0]
print(f'📹 Processing video: {video_path}')
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Processing {total_frames} frames @ {fps:.2f} FPS...')

raw_frames = []
frame_idx = 0

while cap.isOpened() and frame_idx < 600:
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect(mp_image)

    frame_joints = {}
    if result.pose_world_landmarks:
        wl = result.pose_world_landmarks[0]
        l_hip, r_hip = wl[23], wl[24]
        l_sh, r_sh = wl[11], wl[12]
        nose = wl[0]
        l_ear, r_ear = wl[7], wl[8]

        # Hips (Pelvis Center)
        hips_x = (l_hip.x + r_hip.x) * 0.5
        hips_y = -(l_hip.y + r_hip.y) * 0.5 + 1.0
        hips_z = -(l_hip.z + r_hip.z) * 0.5
        frame_joints['Hips'] = {'x': float(hips_x), 'y': float(hips_y), 'z': float(hips_z), 'confidence': 1.0}

        # Chest (Shoulder Center)
        chest_x = (l_sh.x + r_sh.x) * 0.5
        chest_y = -(l_sh.y + r_sh.y) * 0.5 + 1.0
        chest_z = -(l_sh.z + r_sh.z) * 0.5
        frame_joints['Chest'] = {'x': float(chest_x), 'y': float(chest_y), 'z': float(chest_z), 'confidence': 1.0}

        # Spine (Spine Center)
        frame_joints['Spine'] = {
            'x': float((hips_x + chest_x) * 0.5),
            'y': float((hips_y + chest_y) * 0.5),
            'z': float((hips_z + chest_z) * 0.5),
            'confidence': 1.0
        }

        # Head (Ear Midpoint with cranial offset)
        if (l_ear.visibility or 0) > 0.4 and (r_ear.visibility or 0) > 0.4:
            head_x = (l_ear.x + r_ear.x) * 0.5
            head_y = -(l_ear.y + r_ear.y) * 0.5 + 1.0
            head_z = -(l_ear.z + r_ear.z) * 0.5
        else:
            head_x = nose.x
            head_y = -nose.y + 1.0
            head_z = -nose.z

        frame_joints['Head'] = {
            'x': float(head_x),
            'y': float(head_y),
            'z': float(head_z),
            'confidence': float(nose.visibility or 1.0)
        }

        # Neck (Dynamic anatomical cervical bridge smoothly following Head and Spine)
        frame_joints['Neck'] = {
            'x': float(chest_x * 0.50 + head_x * 0.50),
            'y': float(chest_y * 0.50 + head_y * 0.50),
            'z': float(chest_z * 0.50 + head_z * 0.50),
            'confidence': 1.0
        }

        # Nose (Facial gaze landmark)
        frame_joints['Nose'] = {
            'x': float(nose.x),
            'y': float(-nose.y + 1.0),
            'z': float(-nose.z),
            'confidence': float(nose.visibility or 1.0)
        }

        for lm_idx, j_name in LANDMARK_MAP.items():
            lm = wl[lm_idx]
            frame_joints[j_name] = {
                'x': float(lm.x),
                'y': float(-lm.y + 1.0),
                'z': float(-lm.z),
                'confidence': float(lm.visibility or 1.0),
                'visibility': float(lm.visibility or 1.0)
            }

    raw_frames.append({
        'frame_index': frame_idx,
        'timestamp_sec': float(frame_idx) / fps,
        'joints': frame_joints
    })
    frame_idx += 1

cap.release()
print(f'✅ Extraction complete! Processed {len(raw_frames)} frames.')


In [ ]:
# 6. Temporal Motion Regularization (Savitzky-Golay Biomechanical Smoothing)
joint_keys = list(raw_frames[0]['joints'].keys())
T = len(raw_frames)
smoothed_frames = []

if T >= 7:
    window = min(7, T if T % 2 != 0 else T - 1)
    smoothed_data = {}
    for jk in joint_keys:
        coords = np.zeros((T, 3))
        for t in range(T):
            j_info = raw_frames[t]['joints'].get(jk, {'x': 0.0, 'y': 0.0, 'z': 0.0})
            coords[t] = [j_info['x'], j_info['y'], j_info['z']]
        for d in range(3):
            coords[:, d] = savgol_filter(coords[:, d], window_length=window, polyorder=2)
        smoothed_data[jk] = coords

    for t in range(T):
        f_dict = {'frame_index': t, 'timestamp_sec': raw_frames[t]['timestamp_sec'], 'joints': {}}
        for jk in joint_keys:
            f_dict['joints'][jk] = {
                'x': float(smoothed_data[jk][t, 0]),
                'y': float(smoothed_data[jk][t, 1]),
                'z': float(smoothed_data[jk][t, 2]),
                'confidence': 1.0
            }
        smoothed_frames.append(f_dict)
    print(f'✅ Savitzky-Golay temporal smoothing applied (window={window}).')
else:
    smoothed_frames = raw_frames

# 7. Export High-Fidelity JSON Payload for Local Rigging
out_payload = {
    'generator': 'Leocherd Iris Energy Cloud GPU Acceleration 1.0',
    'fps': fps,
    'total_frames': len(smoothed_frames),
    'morphology': 'human',
    'frames': smoothed_frames
}

export_filename = 'iris_kaggle_pose_data.json'
with open(export_filename, 'w') as f:
    json.dump(out_payload, f, indent=2)

print(f'🎉 Success! Pose data saved to: {export_filename}')
print(f'   File size: {os.path.getsize(export_filename) / 1024:.1f} KB')
print('\n👉 Download "iris_kaggle_pose_data.json" from Kaggle Output to your computer!')
